# 자연어 처리 6일차 (2) — Hugging Face `pipeline()` 으로 추론 : 감성 분석 · 문장 생성 · 마켓컬리 리뷰

수업 필기 `nlp_실습9` 를 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재v1 **p.109~111 · 114** — 추론 · Hugging Face 소개 · `pipeline()` · 감성 분석 · 문장 생성 실습)

```python
classifier = pipeline('text-classification', model='모델 이름')   # 토큰화 → 모델 → 후처리를 한 번에
preds = classifier(['문장1', '문장2'])                            # [{'label': ..., 'score': ...}, ...]
```

- **추론(inference)** = 학습이 끝난 모델에 새 입력을 넣어 예측만 하는 것 (가중치 업데이트 없음)
- 함께 받은 `kurly_reviews.csv`(마켓컬리 갈비탕 리뷰 200개 — 교재 p.45 수집 과제)에 영화 리뷰 감성 모델을 적용해 봤습니다 (§4)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다.

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `transformers` · `torch` | `pipeline()` · 모델 실행 | Colab 기본 포함 · 로컬은 `pip install transformers torch` |
| `pandas` · `matplotlib` | 컬리 리뷰 표 · 그래프 | Colab 기본 포함 |

**모델** (첫 실행 때 자동 다운로드 → **인터넷 필요**, 모두 합쳐 약 2GB)

| 작업 | 모델 | 학습 데이터 |
|---|---|---|
| 영어 감성 | `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (pipeline 기본값) | SST-2 영화 리뷰 |
| 영어 금융 감성 | `ProsusAI/finbert` | 금융 뉴스 문장 |
| 한국어 금융 감성 | `snunlp/KR-FinBert-SC` | 한국어 경제 뉴스 |
| 한국어 영화 감성 | `doya/klue-sentiment-nsmc` | 네이버 영화 리뷰(NSMC) |
| 한국어 문장 생성 | `skt/kogpt2-base-v2` | 한국어 위키 · 뉴스 등 |

**데이터** : `kurly_reviews.csv` 를 노트북과 같은 폴더에 (이 폴더의 `data/kurly_reviews.csv`). **실행 순서** : 맨 위부터 **[런타임] → [모두 실행]** — 예측만 하므로 GPU 없이 CPU 로 몇 분.

In [1]:
# ① 설치 (로컬에서 처음 한 번 — Colab 은 기본 포함)
# !pip install -q transformers torch pandas matplotlib

In [2]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'      # 다운로드 진행 막대 숨기기
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()                      # 'LOAD REPORT' 같은 안내 줄 숨기기 (아래 셀에서 직접 확인)
hf_logging.disable_progress_bar()
from huggingface_hub.utils import logging as hub_logging
hub_logging.set_verbosity_error()                     # 'unauthenticated requests' 안내 숨기기 (로그인 없이도 공개 모델은 받아진다)

In [3]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
# Colab(리눅스) : !apt-get install -y fonts-nanum 후 런타임 재시작 → plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [4]:
from transformers import pipeline
import torch
_ = torch.manual_seed(0)

> **📝 고친 부분 (원본 cell 1~3)** `drive.mount(...)` → 이 노트는 드라이브 파일을 쓰지 않는다(모델은 Hugging Face 에서 받음) — 삭제. 리뷰 CSV 는 노트북 폴더에서 읽는다.

## 1. 영어 감성 분석 — 기본 모델

In [5]:
classifier = pipeline(task='text-classification',
                      model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')
text = ['The restaurant is awesome', 'The restaurant is awful']
for t, p in zip(text, classifier(text)):
    print(f"{p['label']:8s} {p['score']:.4f}  {t}")
print('\n레이블 :', classifier.model.config.id2label)

POSITIVE 0.9999  The restaurant is awesome
NEGATIVE 0.9997  The restaurant is awful

레이블 : {0: 'NEGATIVE', 1: 'POSITIVE'}


> **📝 고친 부분 (원본 cell 5)** `pipeline(task='text-classification')` 처럼 모델을 비우면 '**No model was supplied, defaulted to distilbert/…**' 경고와 함께 기본 모델을 고른다. 라이브러리 버전이 바뀌면 기본 모델도 바뀔 수 있으므로 **모델 이름을 적는다**(결과는 원본과 같다).

## 2. 금융 뉴스 감성 — 영어 FinBERT · 한국어 KR-FinBert

In [6]:
fin_classifier = pipeline(task='text-classification', model='ProsusAI/finbert')
for t in ['Stocks rallied and the British pound gained.',
          'The company reported a sharp drop in quarterly profit.',
          'Stocks fell and the British pound weakened.']:
    p = fin_classifier(t)[0]
    print(f"{p['label']:8s} {p['score']:.3f}  {t}")
print('레이블 :', fin_classifier.model.config.id2label)

positive 0.898  Stocks rallied and the British pound gained.
negative 0.976  The company reported a sharp drop in quarterly profit.
negative 0.969  Stocks fell and the British pound weakened.
레이블 : {0: 'positive', 1: 'negative', 2: 'neutral'}


In [7]:
kor_fin_classifier = pipeline(task='text-classification', model='snunlp/KR-FinBert-SC')
news = ['반도체 주도 강세에…코스피 사상 첫 6,600선 넘었다',
        '지난달 120만원대를 웃돌던 삼천당제약 주가가 22일 장중 30만원대까지 떨어졌다.',
        '코스피가 반도체 강세에 힘입어 사상 최고치로 마감했다.',
        '외국인 매수세에 코스피가 큰 폭으로 상승했다.']
for t, p in zip(news, kor_fin_classifier(news)):
    print(f"{p['label']:8s} {p['score']:.3f}  {t}")
print('레이블 :', kor_fin_classifier.model.config.id2label)

neutral  0.909  반도체 주도 강세에…코스피 사상 첫 6,600선 넘었다
negative 0.997  지난달 120만원대를 웃돌던 삼천당제약 주가가 22일 장중 30만원대까지 떨어졌다.
neutral  1.000  코스피가 반도체 강세에 힘입어 사상 최고치로 마감했다.
neutral  0.998  외국인 매수세에 코스피가 큰 폭으로 상승했다.
레이블 : {0: 'negative', 1: 'neutral', 2: 'positive'}


- 원본 결과와 같이 첫 기사(사상 첫 6,600선)는 **neutral 0.909**. 같은 뜻을 '사상 최고치로 마감' · '큰 폭으로 상승' 으로 바꿔 써도 **모두 neutral** — 이 모델은 지수 상승 기사를 중립으로 분류한다. 무엇을 긍정 · 중립으로 볼지는 **모델이 배운 데이터의 기준**을 따르므로, 쓰기 전에 모델 카드와 몇 문장으로 확인해야 한다.
- 레이블 종류와 **번호 순서도 모델마다 다르다** — FinBERT `{0: positive, 1: negative, 2: neutral}`, KR-FinBert `{0: negative, 1: neutral, 2: positive}`, SST-2 모델은 2개.

## 3. 한국어 영화 리뷰 감성 — `LABEL_0` · `LABEL_1` 은 무엇?

In [8]:
kor_classifier = pipeline(task='text-classification', model='doya/klue-sentiment-nsmc')
print('모델에 적힌 레이블 :', kor_classifier.model.config.id2label)

reviews = ['음악이 주가 된, 최고의 음악영화', '절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네']
name = {'LABEL_0': '부정', 'LABEL_1': '긍정'}                 # NSMC : 0 = 부정, 1 = 긍정
for t, p in zip(reviews, kor_classifier(reviews)):
    print(f"{p['label']} ({name[p['label']]}) {p['score']:.4f}  {t}")

모델에 적힌 레이블 : {0: 'LABEL_0', 1: 'LABEL_1'}


LABEL_1 (긍정) 0.9933  음악이 주가 된, 최고의 음악영화
LABEL_0 (부정) 0.9987  절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네


> **📝 고친 부분 (원본 cell 10)** 결과가 `LABEL_1` · `LABEL_0` 으로만 나와 뜻을 알 수 없었다 → 학습 데이터(NSMC)의 규칙 **0 = 부정, 1 = 긍정** 으로 바꿔 읽는다. 직접 미세 조정할 때는 `id2label={0: 'negative', 1: 'positive'}` 를 넣어 두면 처음부터 이름이 나온다([`nsmc_finetune_bert.ipynb`](./nsmc_finetune_bert.ipynb)).
> 예문 `음악이 주가 된, 최고의 음악영화` 는 NSMC **평가용** 데이터 6번째 줄이다.

## 4. 마켓컬리 갈비탕 리뷰 200개에 적용 — 영화 리뷰 모델로 음식 리뷰를?

In [9]:
import pandas as pd
for path in ['kurly_reviews.csv', 'data/kurly_reviews.csv']:
    if os.path.exists(path):
        break
kurly = pd.read_csv(path)
print(kurly.shape, '| 상품 :', kurly['상품명'].unique(), '| 날짜', kurly['날짜'].min(), '~', kurly['날짜'].max())
lens = kurly['리뷰'].str.len()
print('리뷰 글자 수 : 평균', round(lens.mean()), '· 최대', lens.max())
kurly.head(3)

(200, 3) | 상품 : <ArrowStringArray>
['[사미헌] 갈비탕', '[컬리멤버스][사미헌] 갈비탕', '*[사미헌] 갈비탕']
Length: 3, dtype: str | 날짜 2025.11.12 ~ 2026.09.20
리뷰 글자 수 : 평균 171 · 최대 1125


,상품명,날짜,리뷰
0,[사미헌] 갈비탕,2026.09.07,최근 빠르게 2회 구매 했습니다. ^^* 갈비탕하면 사미헌이라고 하길래 구매해 보...
1,[사미헌] 갈비탕,2026.08.31,갈비탕하면 사미헌이라고 하길래 구매해 보았습니다. 아....그런데 저 왜 이 제품...
2,[사미헌] 갈비탕,2026.08.14,"사미헌 갈비탕, 역시 부산 갈비탕은 다르네요 🥹 오늘 컬리에서 같이 산 파김치와 ..."


In [10]:
preds = kor_classifier(kurly['리뷰'].tolist(), truncation=True, max_length=512, batch_size=16)
kurly['감성'] = [name[p['label']] for p in preds]
kurly['확신도'] = [round(p['score'], 3) for p in preds]
print(kurly['감성'].value_counts())

감성
긍정    196
부정      4
Name: count, dtype: int64


In [11]:
pd.set_option('display.max_colwidth', 90)
print('부정으로 나온 리뷰 :')
display(kurly.loc[kurly['감성'] == '부정', ['날짜', '리뷰', '확신도']])

부정으로 나온 리뷰 :


,날짜,리뷰,확신도
78,2026.09.15,아침에 입맛 없을 때 떠껀하게 데워서 먹습니다.,0.630
81,2026.09.14,"수요가 많은 많큼후기도 좋은 사미헌 갈비탕🤩 부모님 드시라고, 배송^^",0.507
135,2026.08.28,갈비탕 끓여서 먹기 전에 대파 썰어넣으면 더 맛있어요.,0.541
152,2026.07.23,잘 뜯기는 고기뼈만 들어있고 육수가 넉넉해서 당면 넣어먹으면 더 맛날것 같아요. 고기 찍어먹는 소스가 들어있으면 더 좋을것 같아요.,0.646


> **📝 고친 부분 (컬리 리뷰 적용)** 긴 리뷰를 그대로 넣으면 BERT 최대 길이(512 토큰)를 넘어 에러가 날 수 있다 → `truncation=True, max_length=512`. 여러 문장은 `batch_size` 로 묶어 빠르게.

- 영화 리뷰로 배운 모델인데도 200개 중 대부분을 긍정으로 잡는다 — '맛있다 · 최고 · 또 살게요' 같은 표현이 영화 리뷰의 긍정 표현과 겹치기 때문.
- 부정으로 나온 리뷰를 직접 읽어 보면 불만이 아니라 '데워서 먹습니다' · '대파 넣으면 더 맛있어요' 같은 **먹는 방법 · 제안**이고, 확신도도 0.5~0.65 로 낮다 → **잘못 분류**한 것. 영화 리뷰에는 이런 문장이 드물다. 분야(영화 → 음식)가 다르면 이렇게 틀리므로, 제대로 쓰려면 음식 리뷰로 **미세 조정**해야 한다.

## 5. 문장 생성 (text generation, 교재 p.114) — KoGPT2

In [12]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
gen_tok = PreTrainedTokenizerFast.from_pretrained('skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>',
                                                  unk_token='<unk>', pad_token='<pad>', mask_token='<mask>')
gen_model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')
print('가중치 :', f'{sum(p.numel() for p in gen_model.parameters()):,}', '개 (GPT-2 small 크기)')

def generate(prompt, n=40, **kw):
    ids = gen_tok(prompt, return_tensors='pt').input_ids
    out = gen_model.generate(ids, max_new_tokens=n, pad_token_id=gen_tok.pad_token_id, **kw)
    return gen_tok.decode(out[0]).split('</d>')[0]       # </d> = 문서 끝 표시 — 그 뒤는 다른 글

for p in ['근육이 커지기 위해서는', '비행기를 타기 위해']:
    print('\n[greedy]', generate(p, do_sample=False, repetition_penalty=2.0))
    torch.manual_seed(0)
    print('[sample]', generate(p, do_sample=True, top_k=50, top_p=0.92, temperature=0.8, repetition_penalty=1.5))

가중치 : 125,164,032 개 (GPT-2 small 크기)



[greedy] 근육이 커지기 위해서는 무엇보다 규칙적인 생활습관이 중요하다.
특히, 아침식사는 단백질과 비타민이 풍부한 과일과 채소를 많이 섭취하는 것이 좋다.
또한 하루 30분 이상 충분한 수면을 취하는 것도 도움이 된다.
아침 식사를 거


[sample] 근육이 커지기 위해서는 무엇보다도 근육을 키우는 것이 중요하다.
근육은 우리 몸에서 가장 큰 비중을 차지한다.
그래서 운동할 때 허리둘레나 골반이 훤히 보이는 ‘척추가 발달된’ 자세를 갖는 것도 좋은 방법이다.




[greedy] 비행기를 타기 위해 공항에 도착하는 순간, 기내에서는 "비행기가 착륙할 때 나는 이륙을 환영한다"는 메시지가 흘러나왔다.
이날 오전 9시30분께 인천공항에서 출발한 에어아시아


[sample] 비행기를 타기 위해 이 같은 계획을 수립했다.
이날 오전 10시30분께 충북 청주공항을 출발한 A380 여객기는 이날 오후 5시경 충주에 도착할 예정이다.


- GPT 는 앞 단어들로 **다음 단어**를 하나씩 이어 붙인다(교재 p.98 '비행기를 타기 위해 □□□').
- `do_sample=False`(greedy)는 항상 가장 확률 높은 단어 → 매번 같은 문장, `do_sample=True` 는 확률대로 뽑아 실행마다 달라진다. `repetition_penalty` 를 주지 않으면 같은 말을 반복하기 쉽다.
- KoGPT2 는 `pipeline('text-generation', model='skt/kogpt2-base-v2')` 만으로는 토크나이저 특수 토큰이 맞지 않아 글자가 깨져 나왔다 → 모델 카드대로 `PreTrainedTokenizerFast` 에 `bos_token='</s>'` 등을 지정한다.
- 생성된 내용은 **사실 확인이 안 된 문장**이다 — 그럴듯해도 틀릴 수 있다.

## 정리

| 작업 | `pipeline` task | 결과 |
|---|---|---|
| 감성 분석 | `'text-classification'` | `[{'label', 'score'}]` — 레이블 뜻은 `model.config.id2label` 과 학습 데이터로 확인 |
| 빈칸 채우기 | `'fill-mask'` | 후보 단어 · 확률 ([`bert_mlm_nsp.ipynb`](./bert_mlm_nsp.ipynb)) |
| 문장 생성 | `'text-generation'` · `model.generate()` | 이어 쓴 문장 |
| 긴 입력 | `truncation=True, max_length=512` | 최대 길이 초과 에러 방지 |